# Why Pydantic?

## The Problem: Python's Dynamic Typing

Python is dynamically typed. This means you can do this:

In [ ]:
def create_user(name, age, email):
    return {"name": name, "age": age, "email": email}


# All of these "work" - but are they correct?
print(create_user("Alice", 30, "alice@example.com"))  # Valid
print(create_user("Bob", "thirty", "not-an-email"))  # Invalid, but Python doesn't care
print(create_user(None, -5, 12345))  # Completely wrong, but no error

**Python won't stop you.** The function accepts anything, and the bugs surface later—often in production.

---

## What Happens Without Pydantic?

### Manual Validation Hell

In [ ]:
MAX_NAME_LENGTH = 100
MAX_AGE = 150


def create_user_validated(name, age, email):
    # You have to write ALL of this yourself
    if not isinstance(name, str):
        raise TypeError("name must be a string")
    if not name.strip():
        raise ValueError("name cannot be empty")
    if len(name) > MAX_NAME_LENGTH:
        raise ValueError("name too long")

    if not isinstance(age, int):
        raise TypeError("age must be an integer")
    if age < 0 or age > MAX_AGE:
        raise ValueError("age must be between 0 and 150")

    if not isinstance(email, str):
        raise TypeError("email must be a string")
    if "@" not in email or "." not in email:
        raise ValueError("invalid email format")

    return {"name": name.strip(), "age": age, "email": email.lower()}


# Try it:
print(create_user_validated("Alice", 30, "alice@example.com"))

In [ ]:
# This will fail with our manual validation
create_user_validated("", -5, "bad")

**Problems:**
- Verbose and repetitive
- Easy to forget edge cases
- Inconsistent across the codebase
- No IDE autocomplete or type hints
- Error messages vary by developer

### Silent Failures

In [ ]:
# API receives this JSON:
data = {"user_id": "abc", "amount": "50.5", "active": "yes"}

# Without validation, you might do:
user_id = data["user_id"]  # Expected int, got string
amount = data["amount"]  # Expected float, got string
active = data["active"]  # Expected bool, got string

print(f"user_id: {user_id} (type: {type(user_id)})")
print(f"amount: {amount} (type: {type(amount)})")
print(f"active: {active} (type: {type(active)})")

In [ ]:
# Later in your code...
total = amount * 2
print(total)

In [ ]:
# And this is even worse - it "works" but is wrong:
if active:  # "yes" is truthy, but so is "no" and "false"!
    print("User is active!")

# Even "no" and "false" are truthy strings!
if "no":
    print("'no' is truthy!")
if "false":
    print("'false' is truthy!")

---

## What Pydantic Gives You

### 1. Declarative Validation

In [ ]:
from pydantic import BaseModel, EmailStr, Field


class User(BaseModel):
    name: str = Field(min_length=1, max_length=100)
    age: float = Field(ge=0, le=150)
    email: EmailStr


# That's it. All the validation from before, in 5 lines.

In [ ]:
# Valid user
user = User(name="Alice", age=30, email="alice@example.com")
print(user)

### 2. Automatic Type Coercion

In [ ]:
# Pydantic intelligently converts types
user = User(name="Alice", age="30", email="alice@example.com")
print(f"age value: {user.age}")
print(f"age type: {type(user.age)}")

### 3. Clear Error Messages

In [ ]:
from pydantic import ValidationError

try:
    User(name="", age="30.5", email="bad")
except ValidationError as e:
    print(e)

### 4. IDE Support & Autocomplete

```python
user = User(name="Alice", age=30, email="alice@example.com")
user.  # IDE shows: name, age, email with types
```

Try it yourself in the cell below:

In [ ]:
user = User(name="Alice", age=30, email="alice@example.com")

# Type user. and press Tab to see autocomplete

### 5. Easy Serialization

In [ ]:
user = User(name="Alice", age=30, email="alice@example.com")

# Convert to dictionary
print("model_dump():")
print(user.model_dump())

print("\nmodel_dump_json():")
print(user.model_dump_json())

### 6. FastAPI Integration

```python
from fastapi import FastAPI

app = FastAPI()

@app.post("/users")
def create_user(user: User) -> User:  # Automatic request validation
    return user                       # Automatic response serialization
```

FastAPI uses Pydantic models for:
- Request body validation
- Response serialization
- OpenAPI documentation generation

---

## Real-World Impact

| Without Pydantic | With Pydantic |
|------------------|---------------|
| Runtime crashes from bad data | Validation errors at the boundary |
| Manual type checking everywhere | Declarative type definitions |
| Inconsistent error messages | Standardized, detailed errors |
| No IDE support for data shapes | Full autocomplete and type hints |
| Manual JSON parsing/serialization | Built-in `model_dump()` / `model_validate()` |
| Documentation gets out of sync | Models ARE the documentation |

---

## When to Use Pydantic

**Use Pydantic when:**
- Receiving external data (APIs, user input, config files)
- Defining domain models with validation rules
- Building FastAPI applications
- Working with structured AI outputs (like Gemini)
- Sharing data shapes across your codebase

**You might skip it for:**
- Simple internal data structures with no validation needs
- Performance-critical hot paths (though Pydantic v2 is fast)

---

## Course Overview

This course teaches Pydantic through 8 hands-on units:

| Unit | Topic | Key Concepts |
|------|-------|--------------|
| 1 | Basic Models | `BaseModel`, types, `model_dump()` |
| 2 | Field Validation | `Field()`, constraints, patterns |
| 3 | Nested Models | Composition, lists, unions |
| 4 | Custom Validators | `@field_validator`, `@model_validator`, `@computed_field` |
| 5 | Model Configuration | `ConfigDict`, aliases, strict mode |
| 6 | Serialization | `model_dump()`, `@field_serializer`, exclude/include |
| 7 | FastAPI Integration | Request/response validation, automatic docs |
| 8 | Gemini Structured Outputs | AI + Pydantic for reliable extraction |